In [1]:
import numpy as np


In [2]:
I  = np.array([[1., 0.],
               [0., 1.]])

J  = np.array([[0., 1.],
               [0., 0.]])

JT = J.T


def laplacian_qtt_cores_dd(d: int):
    """
    Build QTT/MPO cores for the 1D Dirichlet Laplacian Δ_DD^(d),
    i.e. the 2^d x 2^d tridiagonal matrix with 2 on the diagonal
    and -1 on the off-diagonals.

    Returns
    -------
    cores : list[np.ndarray]
        List of MPO cores with shapes:
        first  : (1, 2, 2, 3)
        middle : (3, 2, 2, 3)
        last   : (3, 2, 2, 1)
    """
    if d < 2:
        raise ValueError("Need d >= 2")

    # first core: [ I   J^T   J ]
    W_first = np.zeros((1, 2, 2, 3))
    W_first[0, :, :, 0] = I
    W_first[0, :, :, 1] = JT
    W_first[0, :, :, 2] = J

    # middle core:
    # [ I   J^T   J ]
    # [ 0    J    0 ]
    # [ 0    0   J^T ]
    W_mid = np.zeros((3, 2, 2, 3))
    W_mid[0, :, :, 0] = I
    W_mid[0, :, :, 1] = JT
    W_mid[0, :, :, 2] = J
    W_mid[1, :, :, 1] = J
    W_mid[2, :, :, 2] = JT

    # last core:
    # [ 2I - J - J^T ]
    # [      -J      ]
    # [     -J^T     ]
    W_last = np.zeros((3, 2, 2, 1))
    W_last[0, :, :, 0] = 2 * I - J - JT
    W_last[1, :, :, 0] = -J
    W_last[2, :, :, 0] = -JT

    cores = [W_first] + [W_mid] * (d - 2) + [W_last]
    return cores

def inner_core_product(U: np.ndarray, V: np.ndarray) -> np.ndarray:
    """
    Inner core product:
        (U ⋉ V)_{ab} = sum_g U_{ag} ⊗ V_{gb}

    U shape: (r1, n1, m1, r2)
    V shape: (r2, n2, m2, r3)

    Returns
    -------
    W shape: (r1, n1*n2, m1*m2, r3)
    """
    r1, n1, m1, r2 = U.shape
    r2b, n2, m2, r3 = V.shape
    if r2 != r2b:
        raise ValueError("Bond dimensions do not match")

    W = np.zeros((r1, n1 * n2, m1 * m2, r3))

    for a in range(r1):
        for b in range(r3):
            block = np.zeros((n1 * n2, m1 * m2))
            for g in range(r2):
                block += np.kron(U[a, :, :, g], V[g, :, :, b])
            W[a, :, :, b] = block

    return W


def mpo_to_dense(cores: list[np.ndarray]) -> np.ndarray:
    """Contract a list of MPO cores into a dense matrix."""
    W = cores[0]
    for core in cores[1:]:
        W = inner_core_product(W, core)

    if W.shape[0] != 1 or W.shape[3] != 1:
        raise ValueError("Final contracted object should have trivial boundary bonds")

    return W[0, :, :, 0]

In [3]:
def dense_laplacian_dd(d: int) -> np.ndarray:
    n = 2 ** d
    A = np.zeros((n, n))
    np.fill_diagonal(A, 2.0)
    for i in range(n - 1):
        A[i, i + 1] = -1.0
        A[i + 1, i] = -1.0
    return A


d = 3
cores = laplacian_qtt_cores_dd(d)
A_mpo = mpo_to_dense(cores)
A_ref = dense_laplacian_dd(d)

print(np.allclose(A_mpo, A_ref))   # should be True
print(A_mpo)

True
[[ 2. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  2. -1.  0.  0.  0.  0.  0.]
 [ 0. -1.  2. -1.  0.  0.  0.  0.]
 [ 0.  0. -1.  2. -1.  0.  0.  0.]
 [ 0.  0.  0. -1.  2. -1.  0.  0.]
 [ 0.  0.  0.  0. -1.  2. -1.  0.]
 [ 0.  0.  0.  0.  0. -1.  2. -1.]
 [ 0.  0.  0.  0.  0.  0. -1.  2.]]
